https://arxiv.org/pdf/2005.11401

<h1> Parsing </h1>

<p> 
Consiste em transformar documentos em texto estruturado. <br>
Tem que preservar a estrutura e o contexto do documento para garantir que o mesmo se mantém legível. <br>
É uma excelente oportunidade para adicionar metadados. 
</p>

https://www.thinkevolveconsulting.com/rag-engineers-guide-to-document-parsing/ <br>
https://pypi.org/project/ragparser/ <br>
https://unstructured.io/blog/chunking-for-rag-best-practices

In [ ]:
from unstructured.partition.auto import partition #https://docs.unstructured.io/open-source/core-functionality/partitioning
from pathlib import Path
from langchain_core.documents import Document #https://reference.langchain.com/python/langchain-core/documents

In [ ]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs") #path para pastas de documentos

documentos = [] #lista
id = 0 #idx

for docs in path.iterdir(): #iterar sobre os documentos na pasta
    elementos = partition (str(docs)) #aplicar partition a cada documento

    for x in elementos:
        documentos.append ({
            "id": f"{docs.name}_{id}",
            "page_content": x.text,
            "metadata": {
                "source": docs.name,
                "type": x.category,
                "filetype": x.metadata.to_dict()["filetype"]
            }
        })
        id += 1

'''
elements = partition (path)

for el in elements:
    print (el.text)
    print (el.category)
    print (el.metadata.to_dict())
'''

#display (documentos)
#display (documentos [0]["page_content"])
#display (documentos[2]["metadata"])

In [ ]:
#Como vamos trabalhar com LangChain é necessário converter o parsing para uma classe Document

documentos = [ #Sobreposição de variável para evitar confusão. Esta variável documentos é a variável que sai da fase do Parsing
    Document (
        page_content = x["page_content"], 
        metadata = {
            "id": x["id"],
            "source": x["metadata"]["source"],
            "type": x["metadata"]["type"],
            "file_type": x["metadata"]["filetype"]
        }
    )
    for x in documentos
]

#display (documentos)
#display (documentos[0])
#display (documentos[1].metadata)
#display (documentos[2].page_content)

---------------------------------------------------------------------------

<h1> Chunking </h1>

<p>
Chunking é uma técnica que consiste em converter documentos maiores em unidades mais pequenas (chunks). <br>
Large Language Models têm um problema complicado com context window e por esse motivo não é inteligente e eficiente despejar documentos inteiros nos modelos. <br>
O Chunking acaba por se tornar uma etapa importante na implementação de um sistema RAG porque um Chunk inteligente e eficaz faz toda a diferença.
</p>

https://community.databricks.com/t5/technical-blog/the-ultimate-guide-to-chunking-strategies-for-rag-applications/ba-p/113089

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #https://reference.langchain.com/python/langchain-text-splitters/langchain_text_splitters
#Existem várias opções de text_splitters

In [ ]:
text_splitter = RecursiveCharacterTextSplitter (
    chunk_size = 500, #bom ponto de partida
    chunk_overlap = 75 #10 a 20% do chunk size
)

chunking_documentos = text_splitter.split_documents (documentos)

#Chunks
####display (chunking_documentos)

#Os dois primeiros Chunks
####display (chunking_documentos[0].page_content)
####display (chunking_documentos[1].page_content)

#Estatística dos Chunks
####print (f"Número de Documentos: {len(documentos)} \nNúmero de Chunks: {len(chunking_documentos)}")

---------------------------------------------------------------------------

<h1> Embedding </h1>

<p>
Embedding é a etapa onde convertemos os Chunks em representações vetoriais, capturando assim o seu valor semântico. <br>
Nesta etapa é necessário um Embedding Model.
</p>

https://medium.com/@sharanharsoor/the-complete-guide-to-embeddings-and-rag-from-theory-to-production-758a16d747ac

In [ ]:
page_content = []
#Para o modelo Embedding entra apenas texto, não metadata

for i in range (len(chunking_documentos)):
    page_content.append(chunking_documentos[i].page_content)

#print (page_content)
#print (page_content[0])
#print (page_content[1])

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
emb_model = r"C:\Users\Admin\Desktop\models\Embedding Models"

tokenizer = AutoTokenizer.from_pretrained (emb_model)
modelo = AutoModel.from_pretrained (emb_model)

In [ ]:

#Maneira de fazer Embedding
inputs = tokenizer (page_content, return_tensors = "pt", padding = True, truncation = True) #Padding -> Substitui posições de embeddings não utilizados por [PAD] #Truncation -> Corta quando o número de tokens excede o tamanho máximo do modelo

#print (inputs["input_ids"])
#print (len(inputs["input_ids"]))
#print (tokenizer.decode(inputs["input_ids"][0]))
#print (chunking_documentos[0].page_content)

#Modelo pega em cada token e cria uma representação vetorial chamada de Embedding Dimension (Tamanho depende de cada Embedding Model)
with torch.no_grad():
    outputs = modelo (**inputs) 

#print (outputs)
#print (outputs.keys())
#print (outputs.last_hidden_state)
#print (f"Sequence Length do Primeiro Chunk: {outputs.last_hidden_state[0].shape[0]} \nEmbedding Dimension do Primeiro Chunk: {outputs.last_hidden_state[0].shape[1]}")

#Pooling é o processo de juntar vários vetores (neste caso por token) num único vetor fixo que representa todo o texto.
#Isto é importante porque neste momento temos apenas vetores por cada token do respetivo chunk e nós queremos uma representação final do chunk.
attention_mask = inputs["attention_mask"]
mask = attention_mask.unsqueeze (-1) #Não podemos ter em conta os tokens [PAD]
embeddings = (outputs.last_hidden_state * mask).sum(dim = 1) / mask.sum(dim = 1)

#print (embeddings)
print (f"Texto Chunk 0: {chunking_documentos[0].page_content}\nEmbeddings Chunk 0: {embeddings[0]}")
#print (f"Os {embeddings.shape[0]} Chunks são representados cada, por {embeddings.shape[1]} valores.")


---------------------------------------------------------------------------

<h1> Indexing </h1>

<p>
Após os Embeddings é necessário armazenar os mesmos para posterior consulta. <br>
Para tal usamos Bases de Dados Vetoriais.
</p>


In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Embedding Models")

db = FAISS.from_documents (chunking_documentos, emb_hf) #Este método calcula os Embeddings e mete logo numa base de dados vetorial

#display (db.index.reconstruct(0))
#display (db.docstore._dict)
#display (db.index.ntotal) #Número de Chunks
#display (db.index_to_docstore_id)
#db.save_local("x")

---------------------------------------------------------------------------

https://arxiv.org/pdf/2004.04906

<h1> Retrieval </h1>

<p>
Consiste na capacidade de após receber uma questão (query), conseguir localizar documentos/chunks importantes e relevantes relacionados com a query, na base de dados vetorial.
</p>

<h3> Sparse Retrieval </h3>

<p>
Método clássico. <br>
Procura correspondência direta de palavras, entre a query e os documentos.
</p>

https://huggingface.co/blog/yjoonjang/the-past-and-present-of-sparse-retrieval\

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

In [ ]:
sparse_retrieval = BM25Retriever.from_documents (chunking_documentos, k = 10)

#display (sparse_retrieval)
#display (sparse_retrieval.docs)

<h3> Dense Retrieval </h3>

<p>
Neste tipo de Retrieval ambas as queries e os documentos são transformados em embeddings. <br>
A relevância é calculada usando métricas de similiaridade (produto interno ou similaridade do cosseno) entre os embeddings da query e os embeddings dos documentos.
</p>

https://www.emergentmind.com/topics/dense-retrieval-dr

In [ ]:
###Exemplo

#Conversão da query em Embeddings
query = "Quem venceu a primeira Copa do Mundo?"

embededed_query = emb_hf.embed_query (query)
#display (embededed_query)

#simil = db.similarity_search (query)
#simil_scores = db.similarity_search_with_relevance_scores (query)

#simil_vector = db.similarity_search_by_vector (embededed_query, k = 10)
#simil_vector_scores = db.similarity_search_with_score_by_vector (embededed_query, k = 10)

[Document(id='29b5e65c-22c2-4704-9627-2fb2ecdd8e97', metadata={'id': 'Mundial de Futebol.txt_1', 'source': 'Mundial de Futebol.txt', 'type': 'NarrativeText', 'file_type': 'text/plain'}, page_content='Na final, o Uruguai derrotou a Argentina por 4–2 na frente de quase 93 mil pessoas em Montevidéu e se tornou a primeira seleção a vencer a Copa do Mundo. Após a criação da Copa do Mundo, a FIFA e o COI discordaram sobre o status de jogadores amadores e, assim, o futebol foi retirado dos Jogos Olímpicos de Verão de 1932, retornando nos Jogos Olímpicos de Verão de 1936, porém ofuscado pela mais prestigiada Copa do Mundo. Os problemas enfrentados pelos primeiros torneios da Copa do Mundo foram as'),
 Document(id='11fb8a5c-c52b-40a2-bd77-115cc14d4d4d', metadata={'id': 'Mundial de Futebol.txt_1', 'source': 'Mundial de Futebol.txt', 'type': 'NarrativeText', 'file_type': 'text/plain'}, page_content='busca trazer pessoas de todo o mundo. Se você não dá a possibilidade de participar, elas não melho

In [ ]:
#Com LangChain é possível converter a Vector Database diretamente para um Dense Retrieval

dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

retrieval = dense_retrieval.invoke (query) #== db.similarity_search_by_vector (embededed_query, k = 10)
#retrieval

[Document(id='29b5e65c-22c2-4704-9627-2fb2ecdd8e97', metadata={'id': 'Mundial de Futebol.txt_1', 'source': 'Mundial de Futebol.txt', 'type': 'NarrativeText', 'file_type': 'text/plain'}, page_content='Na final, o Uruguai derrotou a Argentina por 4–2 na frente de quase 93 mil pessoas em Montevidéu e se tornou a primeira seleção a vencer a Copa do Mundo. Após a criação da Copa do Mundo, a FIFA e o COI discordaram sobre o status de jogadores amadores e, assim, o futebol foi retirado dos Jogos Olímpicos de Verão de 1932, retornando nos Jogos Olímpicos de Verão de 1936, porém ofuscado pela mais prestigiada Copa do Mundo. Os problemas enfrentados pelos primeiros torneios da Copa do Mundo foram as'),
 Document(id='11fb8a5c-c52b-40a2-bd77-115cc14d4d4d', metadata={'id': 'Mundial de Futebol.txt_1', 'source': 'Mundial de Futebol.txt', 'type': 'NarrativeText', 'file_type': 'text/plain'}, page_content='busca trazer pessoas de todo o mundo. Se você não dá a possibilidade de participar, elas não melho

<h3> Reciprocal Rank Fusion (RRF) </h3>

<p>
É um método usado para combinar scores. <br>
Muito usado em sistemas Hybrid Retrieval tal como este. <br>
Consiste na criação de um sistema de Ranking entre métodos de Retrieval. 
</p>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + rank_i(d)} 
$$

CONTINUAR!
TENTAR PERCEBER QUAL É A MELHOR MANEIRA

In [ ]:
from collections import defaultdict

def Reciprocal_Rank_Fusion (sparse, dense, k = 60):
    
    rrf = {}

    for scores, doc in enumerate(sparse):
        rrf[doc.id] += 1 / (k + scores + 1)

    for scores, doc in enumerate(dense):
        rrf[doc.id] += 1 / (k + scores + 1)

    return sorted(rrf.items(), key = lambda x: x[1], reverse = True) #Lambda cria uma mini função

In [149]:
def rrf(rankings, k = 60):
    scores = {}

    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            doc_id = doc.page_content

            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)

    return sorted(scores.items(), key = lambda x: x[1], reverse = True)

In [151]:
###Exemplo

query = "Quem venceu a primeira Copa do Mundo?"

sparse = sparse_retrieval.invoke (query)
dense = dense_retrieval.invoke (query)

teste = rrf ([sparse, dense])

teste

[('Na final, o Uruguai derrotou a Argentina por 4–2 na frente de quase 93 mil pessoas em Montevidéu e se tornou a primeira seleção a vencer a Copa do Mundo. Após a criação da Copa do Mundo, a FIFA e o COI discordaram sobre o status de jogadores amadores e, assim, o futebol foi retirado dos Jogos Olímpicos de Verão de 1932, retornando nos Jogos Olímpicos de Verão de 1936, porém ofuscado pela mais prestigiada Copa do Mundo. Os problemas enfrentados pelos primeiros torneios da Copa do Mundo foram as',
  0.032266458495966696),
 ('uma. A Argentina é tricampeã, ganhando em (1978, 1986 e 2022), e é a atual vencedora, enquanto que a equipe que venceu a primeira edição, o Uruguai, conquistou duas vezes, em (1930 e 1950), assim como a França (1998 e 2018). Finalmente, a Inglaterra (1966) e a Espanha (2010) ganharam uma Copa do Mundo cada uma. O Uruguai (1930), a Itália (1934), a Inglaterra (1966), a Alemanha (1974), a Argentina (1978) e a França (1998) conseguiram vencer ao menos uma edição em c